In [1]:
from google.colab import userdata
!pip install langchain-openai
import langchain
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 1.7 MB/s eta 0:00:00


In [2]:
#@title Install Packages and Setup Environment { display-mode: "form" }

from google.colab import files
import zipfile
import io
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt

# potential solution
!pip install protobuf==3.20 &> /dev/null
!pip install tensorflow==2.8 &> /dev/null
!apt install --allow-change-held-packages libcudnn8=8.1.0.77-1+cuda11.2 &> /dev/null


!pip -q install streamlit &> /dev/null
!pip -q install pyngrok &> /dev/null
from pyngrok import ngrok

#not necessary for now
#!pip install tensorflowjs
#import tensorflowjs as tfjs

import json
from urllib.request import urlretrieve

from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
from IPython.display import Image
import os
import gdown

def take_photo(filename='photo.jpg', quality=0.8):
  js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = 'Capture';
      div.appendChild(capture);

      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      // Resize the output to fit the video element.
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

      // Wait for Capture to be clicked.
      await new Promise((resolve) => capture.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
  display(js)
  data = eval_js('takePhoto({})'.format(quality))
  binary = b64decode(data.split(',')[1])
  with open(filename, 'wb') as f:
    f.write(binary)
  return filename

def capture_image(filename):
  try:
    take_photo(filename)
    print('Saved to {}'.format(filename))

    # Show the image which was just taken.
    display(Image(filename))
  except Exception as err:
    # Errors will be thrown if the user does not have a webcam or if they do not
    # grant the page permission to access it.
    print(str(err))

import numpy as np
import tensorflow as tf
import cv2
from google.colab.patches import cv2_imshow

# Ensure comptability with different TF versions
version_fn = getattr(tf.keras, "version", None)
if version_fn and version_fn().startswith("3."):
  import tf_keras as keras
else:
  keras = tf.keras

import warnings
warnings.filterwarnings('ignore')


In [3]:
#@title Upload and extract model into Colab

# upload model zip file
uploaded = files.upload()

# extract model
try:
  file_name = list(uploaded.keys())[-1]
  print("Extracting model...")
  with zipfile.ZipFile(file_name, 'r') as zip_ref:
      zip_ref.extractall('')

  class_names = np.genfromtxt("labels.txt", dtype="str", delimiter='\n')
  for i in range(len(class_names)):
    class_names[i] = ' '.join(class_names[i].split(' ')[1::])

  predictions = [0.0] * len(class_names)

  model = keras.models.load_model('keras_model.h5', compile=False)
  print("Success! Model Extracted!")
except (IndexError, NameError):
  print("Oops! Cannot find file to unzip. Please try uploading a zip file for your model")

Saving converted_keras (1).zip to converted_keras (1).zip
Extracting model...
Success! Model Extracted!


In [22]:
 ok_model=ChatOpenAI(
    api_key=userdata.get('Franc'),
    base_url='https://open.bigmodel.cn/api/paas/v4',
    model='glm-4',
    max_completion_tokens=100
  )


In [23]:
embeddings= OpenAIEmbeddings(
    api_key=userdata.get("Franc"),
    base_url="https://open.bigmodel.cn/api/paas/v4",
    model="embedding-3",
)

In [7]:
prompt2= """Yuu are a helpfull assistant for farmers,
 you will be receiving a name of the maize disease and use the given context to answer how does the disease affect
 and the ways to avoid it from maize.Provide only essential symptoms and treatment recommendations for {Disease} in maize. Keep it concise.
 ***Disease***
  {Disease}
 ***Answer***

 When you recieve the disease you will be answering like this;
 Disease_name:
 Effects:
 Solution:

 """

In [14]:
prompts="""Answer the following questions as best as you can. You have access to the following tools:
{tools}. You will be receiving an image, after you receive an image take it to the tool, Find_disease(image_path) for finding the disease and the output of the
tool is the name known as predicted_class_name which you should take to the tool, Search_for_disease_details(predicted_class_name) for finding the disease details.
The output from that tool is the disease details and it is your response which is the best answer to the user.

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: Analyze the image of a maize leaf showing signs of disease. Identify the specific disease affecting the maize plant, and provide a brief description of its symptoms. Then, recommend effective treatment or prevention methods suitable for smallholder farmers. Respond clearly and concisely.
{agent_scratchpad}
"""

In [9]:
from langchain.agents import create_react_agent, AgentExecutor
from langchain.tools import tool

In [26]:
import os
# image_path = "maize_leaf_image.jpg"
# print("Full image path:", os.path.abspath(image_path))
# img = cv2.imread(os.path.abspath(image_path))
import tensorflow as tf

In [ ]:

import os
import cv2
import numpy as np
import tensorflow as tf
import gradio as gr
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain.agents import create_react_agent, AgentExecutor
from langchain.tools import tool

# Tools
@tool
def Search_for_disease_details(predicted_class_name):
    """Returns essential info about the maize disease."""
    prompt_template = PromptTemplate.from_template(template=prompt2)
    chain = LLMChain(llm=ok_model, prompt=prompt_template)
    response = chain.invoke(input={'Disease': predicted_class_name})
    return response['text']


@tool
def Find_disease(image_path: str) -> str:
    """Gets image from executor and runs prediction to identify maize leaf disease."""
    print(f"🧾 Received image path: {image_path}")

    if not os.path.exists(image_path):
        raise FileNotFoundError(f"❌ File not found at: {image_path}")

    try:
        image = tf.io.read_file(image_path)
        image = tf.image.decode_image(image, channels=3, expand_animations=False)
        image = tf.image.resize(image, [224, 224])
        image = image / 255.0
        image = tf.expand_dims(image, axis=0)

        predictions = model.predict(image)
        score = tf.nn.softmax(predictions[0])

        predicted_class_index = np.argmax(score)

        predicted_class_name = class_names[predicted_class_index]

        confidence = float(np.max(score))

        prompt_template = PromptTemplate.from_template(template=prompt2)
        chain = LLMChain(llm=ok_model, prompt=prompt_template)
        response = chain.invoke(input={'Disease': predicted_class_name})
        return response['text']

    except Exception as e:
        raise ValueError(f"❌ TensorFlow failed to process the image. Error: {str(e)}")



tools = [Find_disease, Search_for_disease_details]
prot = PromptTemplate.from_template(prompts)
agent = create_react_agent(ok_model, tools=tools, prompt=prot)
executor = AgentExecutor(agent=agent, tools=tools, verbose=True, )


def process_image(image_path: str) -> str:
    """Passes the image path to the Find_disease tool and returns the predicted disease."""
    print(f"🧾 Received image path: {image_path}")

    try:
        response = Find_disease.invoke({"image_path": image_path})
        return response
    except Exception as e:
        raise RuntimeError(f"Failed to process image with Find_disease. Error: {str(e)}")


def ui():
    chat_ui = gr.Interface(
        fn=process_image,
        inputs=gr.Image(type="filepath"),
        outputs=gr.TextArea(label="Disease Information"),
        title="FRANCIS GOOD MAIZE AI🌽🌽",
        description="Upload a maize leaf image to detect disease and get actionable insights 🌿🧠"
    )
    chat_ui.launch(debug=True)


if __name__ == "__main__":
    ui()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://13afecc66aa4549af9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


🧾 Received image path: /tmp/gradio/80f6ded1ff700728dbc22638f69a80b9622a44a5730804a9763cda3c565b0d8a/imgi_13_default.jpg
🧾 Received image path: /tmp/gradio/80f6ded1ff700728dbc22638f69a80b9622a44a5730804a9763cda3c565b0d8a/imgi_13_default.jpg
1/1 [==============================] - 0s 59ms/step
